# MUC01 Category Intelligence – Retail Product Performance EDA




**Author:** Maganti Nagarjuna  

**Project:** Retail Product Performance Exploratory Data Analysis (EDA)  
**Analysis Period:** January–June 2026

## 1. CBIM Problem Canvas

### Situation (S)
A regional retail chain operates 12 stores across Andhra Pradesh, selling products across five categories: Electronics, Apparel, Grocery, Home & Kitchen, and Personal Care. The Category Manager is preparing for an upcoming quarterly supplier review and needs a clear, data-backed understanding of category and product performance to evaluate shelf-space allocation and promotional investment.

### Complication (C)
The Category Manager lacks a clear, data-backed understanding of which product categories and individual products are performing well or declining over time. Without reliable evidence on product performance and discount effectiveness, she may struggle to justify shelf-space allocation, promotional investment, and stocking decisions during the supplier review meeting.

### Question (Q)
Which product categories are growing or declining, which products are the top and bottom performers within each category, and are higher discounts generating sufficient revenue to justify their use? The analysis will help the Category Manager make data-driven decisions about stocking levels, shelf-space allocation, promotional investment, and which suppliers to prioritise or challenge during the review meeting.

### SSOT (Single Source of Truth)
The single source of truth is **MUC01_Retail_Sales_Dataset.csv**, containing retail transaction data for January to June 2026. All performance analysis and business recommendations in this notebook are based on this dataset.

### Success Definition
The analysis will be useful if it provides clear, data-backed insights into category performance, top and bottom-performing products, and discount effectiveness. A successful output will help the Category Manager confidently present findings during the supplier review, justify shelf-space and promotional investment decisions, and identify suppliers to prioritise or challenge based on actual sales performance.

### Primary Stakeholder
The primary stakeholder is the **Category Manager**, who needs data-backed insights into category and product performance to decide which suppliers deserve continued or increased shelf space and promotional investment, and which suppliers should be challenged based on poor performance.



## 6.1  Load and inspect the data
#### Load the dataset and get oriented. Check the shape, data types, and null values. Run a .describe() summary and write 2 observations from it - not what the numbers are, but what they might mean for a retail business. How many unique products are there? How many unique categories? Write these as markdown observations, not just code output.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Load the dataset
df = pd.read_csv("MUC01_Retail_Sales_Dataset.csv")
print("Dataset shape:",df.shape)

# 2. Check data types and non-null counts
print("\nDataset information:")
df.info()

# 3. Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# 4. Statistical summary
print("\nSummary statistics:")
print(df.describe())

# 5. Number of unique products
print("\nUnique products:",df["product_name"].nunique())

# 6. Number of unique categories
print("Unique categories:",df["category"].nunique())

# 7. Check duplicate rows
print("\nDuplicate rows:",df.duplicated().sum())

# 8. Check duplicate transaction IDs
print("Duplicate transaction IDs:",df["transaction_id"].duplicated().sum())


### 6.1: Load and Inspect the Data

The retail sales dataset contains **107,836 transactions and 10 columns**, covering six months of retail sales data from January to June 2026. I inspected the data types, checked for missing values, and reviewed the statistical summary to understand the dataset before performing further analysis.

**Observations:**

1. **Product Pricing:** The average unit price is approximately ₹1,778, while the median is ₹488.45. This substantial difference, along with the wide variation in prices, suggests that a relatively small number of expensive products may be increasing the average. The retailer should analyse product-level revenue to understand which products contribute most to sales.

2. **Revenue Distribution:** The average revenue per transaction is approximately ₹4,263.92, compared with a median of ₹1,034.62. This indicates that transaction values vary considerably, with some high-revenue transactions potentially pulling the average upward. Identifying the products and categories behind these transactions could help the Category Manager understand the drivers of retail revenue.

**Unique Products and Categories:**

The dataset contains **25 unique products across 5 product categories**. This provides an opportunity to compare product performance within each category and identify the products contributing most and least to total revenue.


### 6.2  Category revenue trend over time
Group revenue by category and month and plot a line chart showing each category's monthly revenue trend - all five categories on one chart. This is the most important chart in the analysis for the Category Manager.


In your markdown cell, answer: which category has shown the steepest decline over the six months, and which has been the most consistent performer? Frame your answer as you would to the Category Manager - not as a data observation, but as a business signal she should act on.

In [ ]:

df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)

# Calculate monthly revenue for each category
monthly_revenue = df.groupby(["month", "category"])["revenue"].sum().reset_index()
revenue_pivot = monthly_revenue.pivot(index="month",columns="category",values="revenue")
print(revenue_pivot)
plt.figure(figsize=(12, 6))
for category in revenue_pivot.columns.unique():
    plt.plot(revenue_pivot.index,revenue_pivot[category],marker="o",label=category)
plt.title("Monthly Revenue Trend by Product Category")
plt.xlabel("Month")
plt.ylabel("Total Revenue (₹)")
plt.legend(title="Product Category")
plt.grid(True, alpha=0.3)
plt.ticklabel_format(axis="y",style="plain",useOffset=False)
plt.tight_layout()
plt.show()

# Check monthly revenue consistency by category
monthly_std = revenue_pivot.std()
cv = (monthly_std / revenue_pivot.mean()) * 100
print("Monthly Revenue Coefficient of Variation (%):")
print(cv.sort_values())
print("\nMost consistent category:")
print(cv.idxmin())

### 6.2 Category Revenue Trend Over Time

Electronics has shown the steepest decline in revenue over the six months, falling from approximately ₹4.57 crore in January to ₹2.97 crore in June, a drop of about 35%. This declining trend raises concerns about future performance and weakens the case for increasing shelf-space allocation or promotional investment. The Category Manager should investigate the reasons for this decline and ask the supplier to present a recovery plan before committing further resources.

Grocery is the most consistent performer, with a coefficient of variation of just 3.87%, indicating relatively stable monthly revenue. This predictable performance supports maintaining its current shelf-space allocation and focusing on sustaining sales. The Category Manager should prioritise Grocery's stability while challenging the declining Electronics category to justify future investment.



### 6.3  Top and bottom products by category
For each of the five categories, identify the top 2 and bottom 2 products by total revenue over the six months. Present this as a summary table. Then write a markdown cell answering: if you were advising the Category Manager on which supplier to challenge in the review meeting, which category would you point to and why?

In [ ]:
# Calculate total revenue for each product in each category
product_revenue = (df.groupby(["category", "product_name"])["revenue"].sum().reset_index())

# Identify the top 2 and bottom 2 products in each category
results = []
for category in product_revenue["category"].unique():
    category_data = product_revenue[product_revenue["category"] == category].sort_values("revenue", ascending=False)
    top_2 = category_data.head(2).copy()
    top_2["performance"] = "Top 2"
    bottom_2 = category_data.tail(2).sort_values("revenue", ascending=True).copy()
    bottom_2["performance"] = "Bottom 2"
    results.append(top_2)
    results.append(bottom_2)
    
# Combine the results into one summary table
summary_table = pd.concat(results, ignore_index=True)
print(summary_table[["category", "product_name", "performance", "revenue"]])

### 6.3 Top and Bottom Products by Category

I would recommend challenging the Electronics category supplier because its overall revenue declined over the six months. Within this category, Smart TV generated ₹4.19 crore and Tablet generated ₹4.47 crore, making them the two lowest-revenue products in the category. During the supplier review meeting, I would ask the supplier to explain their performance and propose a recovery plan. I would also review product demand and promotional effectiveness before deciding whether these products need revised promotions or changes in shelf-space allocation. This would help the Category Manager make evidence-based decisions about supplier performance and future investment.

Within each category the product revenues are close together, so the Electronics case rests mainly on the Step 2 trend rather than on the product ranking alone. The dataset has no supplier column, so each category's supplier is treated as one group.


## 6.4  Discount effectiveness analysis
Group transactions by discount level - 0%, 5%, 10%, 15%, 20%, and 25%+ - and calculate the average revenue per transaction and average units sold for each level. Plot a bar chart showing average revenue per transaction by discount bracket.


In your markdown, answer: are higher discounts generating meaningfully more revenue per transaction, or are they simply cutting margin without a proportionate return? Take a clear position and back it with the numbers.


In [ ]:
# Create discount analysis
df["discount_bracket"] = pd.cut(df["discount_pct"],bins=[-0.1, 0, 5, 10, 15, 20, 30],labels=["0%", "5%", "10%", "15%", "20%", "25%+"])

# Calculate average revenue, average units sold, and transaction count
discount_analysis = df.groupby("discount_bracket", observed=False).agg(
avg_revenue=("revenue", "mean"),avg_units_sold=("units_sold", "mean"),transaction_count=("transaction_id", "count")).reset_index()
display(discount_analysis)

# Plot only discount brackets with available data
plot_data = discount_analysis.dropna(subset=["avg_revenue"])
plt.figure(figsize=(10, 6))
plt.bar(plot_data["discount_bracket"].astype(str),plot_data["avg_revenue"])
plt.title("Average Revenue per Transaction by Discount Level")
plt.xlabel("Discount Bracket")
plt.ylabel("Average Revenue per Transaction (₹)")
plt.ticklabel_format(axis="y", style="plain", useOffset=False)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Report brackets with no transactions
empty_brackets = discount_analysis.loc[discount_analysis["avg_revenue"].isna(),"discount_bracket"].astype(str).tolist()
if empty_brackets:
    print("No transactions found for:", ", ".join(empty_brackets))

### 6.4 Discount Effectiveness Analysis

**Position: the observed results do not show a proportionate return from deeper discounts.** Average revenue per transaction generally decreases as the discount increases, although this observational analysis cannot prove that discounts caused the change. Transactions with no discount have the highest average revenue of ₹4,685.65, while the average decreases to ₹4,303.73 at a 5% discount and ₹4,005.55 at a 10% discount. At a 15% discount, average revenue increases slightly to ₹4,069.16, but it falls further to ₹3,631.73 at a 20% discount. Meanwhile, average units sold remain relatively stable, ranging from 2.57 to 2.60 units across these discount levels.

This suggests that higher discounts are not associated with a proportionate increase in units sold or average revenue in this dataset. I recommend reviewing deeper discounts with suppliers and testing whether they produce incremental sales. Revenue is recorded after discount, and the lack of a clear increase in units alongside lower revenue per transaction may indicate margin pressure; however, margin erosion cannot be measured or causally attributed without product cost and profit-margin data.

**Data Limitation:** The dataset's maximum recorded discount is 20%, so there are no transactions in the 25%+ discount bracket. Therefore, no conclusion can be drawn about discounts of 25% or more.


## 6.5  Day-of-week and monthly patterns
Extract the day of week from the date column and plot average daily revenue by day. Then plot total revenue by month. Identify the best and worst day and month for the chain overall, then do the same for the single most interesting category from Step 5.2. Does that category follow the same pattern as the chain, or behave differently? Write 2–3 sentences on what this means for promotional timing.

In [ ]:

# Convert date column
df["date"] = pd.to_datetime(df["date"])
df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.to_period("M").astype(str)
day_order = ["Monday", "Tuesday", "Wednesday","Thursday", "Friday", "Saturday", "Sunday"]

# 1. Average daily revenue by day of week - overall chain
daily_sales = (df.groupby(["date", "day_of_week"])["revenue"].sum().reset_index())
day_revenue = (daily_sales.groupby("day_of_week")["revenue"].mean().reindex(day_order))
print("Average Daily Revenue by Day of Week:")
print(day_revenue.round(2))
print("Best day:", day_revenue.idxmax())
print("Worst day:", day_revenue.idxmin())
# Plot 1
plt.figure(figsize=(10, 6))
plt.bar(day_revenue.index, day_revenue.values)
plt.title("Average Daily Revenue by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Daily Revenue (₹)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Total revenue by month - overall chain
monthly_revenue = df.groupby("month")["revenue"].sum().sort_index()
print("\nTotal Monthly Revenue - Overall Chain:")
print(monthly_revenue.round(2))
print("Best month:", monthly_revenue.idxmax())
print("Worst month:", monthly_revenue.idxmin())
# Plot 2
plt.figure(figsize=(10, 6))
plt.bar(monthly_revenue.index, monthly_revenue.values)
plt.title("Total Monthly Revenue - Overall Chain")
plt.xlabel("Month")
plt.ylabel("Total Revenue (₹)")
plt.ticklabel_format(axis="y", style="plain", useOffset=False)
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Repeat the analysis for Electronics
electronics_df = df[df["category"] == "Electronics"]

# Electronics average daily revenue by day
electronics_daily_sales = (electronics_df.groupby(["date", "day_of_week"])["revenue"].sum().reset_index())
electronics_day_revenue = (electronics_daily_sales.groupby("day_of_week")["revenue"].mean().reindex(day_order))
print("\nElectronics - Average Daily Revenue by Day:")
print(electronics_day_revenue.round(2))
print("Best day:", electronics_day_revenue.idxmax())
print("Worst day:", electronics_day_revenue.idxmin())

# Electronics total revenue by month
electronics_monthly = (electronics_df.groupby("month")["revenue"].sum().sort_index())

# Plot Electronics average daily revenue by weekday
plt.figure(figsize=(10, 6))
plt.bar(electronics_day_revenue.index, electronics_day_revenue.values)
plt.title("Electronics: Average Daily Revenue by Weekday")
plt.xlabel("Day of Week")
plt.ylabel("Average Daily Revenue (₹)")
plt.xticks(rotation=45)
plt.ticklabel_format(axis="y", style="plain", useOffset=False)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nElectronics - Total Monthly Revenue:")
print(electronics_monthly.round(2))
print("Best month:", electronics_monthly.idxmax())
print("Worst month:", electronics_monthly.idxmin())
# Plot Electronics total revenue by month
plt.figure(figsize=(10, 6))
plt.bar(electronics_monthly.index, electronics_monthly.values)
plt.title("Electronics: Total Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Total Revenue (₹)")
plt.ticklabel_format(axis="y", style="plain", useOffset=False)
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 Day-of-Week and Monthly Patterns

**Observations:**

1. **Day-of-Week Pattern:** The overall retail chain generates the highest average daily revenue on Saturday (₹35.24 lakh) and the lowest on Monday (₹20.12 lakh). Electronics follows the same pattern, with its highest average daily revenue on Saturday (₹17.87 lakh) and lowest on Monday (₹9.71 lakh).

2. **Monthly Revenue Pattern:** The overall chain records its highest monthly revenue in January 2026 (₹8.48 crore) and its lowest in June 2026 (₹6.93 crore). Electronics also records its highest revenue in January (₹4.57 crore) and its lowest in June (₹2.97 crore), following the same monthly pattern as the overall chain.

**Business Recommendation:** 

Since Monday has the lowest average daily revenue for both the overall chain and Electronics, the retailer could test targeted Monday promotions to stimulate demand. Saturday already generates the highest revenue, so discounts should not be offered automatically on that day; instead, the retailer should evaluate whether promotions generate additional sales beyond normal demand. The substantial decline in Electronics revenue from January to June should also be investigated before deciding on further promotional campaigns.


### 6.6  Your recommendation to the Category Manager
Close your notebook with a markdown cell titled "Category Manager Briefing." Write 4–6 sentences that you would actually say to the Category Manager walking into the supplier review. Cover:
•	Which category has the strongest case for increased investment and why.
•	Which category or product you would challenge in the supplier meeting - with a specific revenue figure.
•	One finding from the discount analysis that the Category Manager should raise with suppliers.
•	What one additional piece of data you would want before making a final recommendation.

### Category Manager Briefing

I recommend investing more in Apparel, whose monthly revenue grew 34.1%, from approximately ₹1.00 crore in January to ₹1.35 crore in June. I would challenge the Electronics supplier: monthly revenue fell about 35%, from ₹4.57 crore to ₹2.97 crore, and its two weakest products over six months are Smart TV (₹4.19 crore) and Tablet (₹4.47 crore). I would ask the supplier for a recovery plan before any extra shelf space or promotional spend. On discounts, average revenue per transaction falls from ₹4,685.65 with no discount to ₹3,631.73 at 20%, while units sold stay nearly unchanged; this does not show a proportionate return from deeper discounts. Since revenue is already after discount, this may indicate margin pressure, but cost and margin data are needed to measure profitability or establish causation. Before finalising, I would want product cost and profit margin data.